In [ ]:
!pip install geopandas


SyntaxError: invalid syntax (3219303094.py, line 2)

In [1]:
import pandas as pd

# Load directly from NY Open Data (no download needed)
url = "https://data.ny.gov/api/views/2e6c-s6fp/rows.csv?accessType=DOWNLOAD"
df = pd.read_csv(url)

# Preview the data
print(df.shape)      # how many rows & columns
print(df.head())     # first 5 rows
print(df.columns)    # list all column names

(4918, 65)
                                            the_geom        GEOID  \
0  MULTIPOLYGON (((-73.80645699999998 40.71205599...  36081044800   
1  MULTIPOLYGON (((-73.79363599999998 40.71381899...  36081045800   
2  MULTIPOLYGON (((-73.79202899999999 40.71106899...  36081046200   
3  MULTIPOLYGON (((-73.87468499999999 40.74334899...  36081046300   
4  MULTIPOLYGON (((-73.79187099999999 40.71379299...  36081046400   

         DAC_Designation           REDC  County      City_Town NYC_Region  \
0  Not Designated as DAC  New York City  Queens  New York city        NYC   
1  Not Designated as DAC  New York City  Queens  New York city        NYC   
2      Designated as DAC  New York City  Queens  New York city        NYC   
3      Designated as DAC  New York City  Queens  New York city        NYC   
4  Not Designated as DAC  New York City  Queens  New York city        NYC   

  Urban_Rural Tribal_Designation Household_Low_Count_Flag  ...  \
0       urban                 No             

In [ ]:
import pandas as pd
import geopandas as gpd
import folium
import json
from pathlib import Path
from shapely import wkt
from shapely.geometry import mapping, Point

base = Path().resolve()
while not (base / "readme.md").exists() and base != base.parent:
    base = base.parent

# ── 1. Load NYCHA ─────────────────────────────────────────────────────────────
nycha = pd.read_csv(base / "data/processed/nycha_residential.csv")
nycha = nycha.dropna(subset=['latitude', 'longitude'])

# ── 2. Load DAC ───────────────────────────────────────────────────────────────
dac_url = "https://data.ny.gov/api/views/2e6c-s6fp/rows.csv?accessType=DOWNLOAD"
dac = pd.read_csv(dac_url, storage_options={"verify": False})
dac_nyc = dac[
    (dac['NYC_Region'] == 'NYC') &
    (dac['DAC_Designation'] == 'Designated as DAC')
].copy()

# ── 3. Convert both to GeoDataFrames ─────────────────────────────────────────
# NYCHA points
nycha_gdf = gpd.GeoDataFrame(
    nycha,
    geometry=gpd.points_from_xy(nycha['longitude'], nycha['latitude']),
    crs='EPSG:4326'
)

# DAC polygons from WKT
dac_nyc['geometry'] = dac_nyc['the_geom'].apply(wkt.loads)
dac_gdf = gpd.GeoDataFrame(dac_nyc, geometry='geometry', crs='EPSG:4326')

# ── 4. Spatial join — check which NYCHA points fall inside a DAC polygon ──────
joined = gpd.sjoin(nycha_gdf, dac_gdf[['geometry']], how='left', predicate='within')
nycha_gdf['in_dac'] = ~joined['index_right'].isna()

print(f"Total NYCHA buildings:      {len(nycha_gdf)}")
print(f"NYCHA buildings in DAC:     {nycha_gdf['in_dac'].sum()}")
print(f"NYCHA buildings NOT in DAC: {(~nycha_gdf['in_dac']).sum()}")

# ── 5. Build GeoJSON from DAC polygons ────────────────────────────────────────
features = []
for _, row in dac_gdf.iterrows():
    try:
        features.append({
            "type": "Feature",
            "geometry": mapping(row['geometry']),
            "properties": {}
        })
    except:
        pass
dac_geojson = {"type": "FeatureCollection", "features": features}
print(f"DAC tract polygons loaded:  {len(features)}")

# ── 6. Build map ──────────────────────────────────────────────────────────────
m = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB positron')

folium.GeoJson(
    dac_geojson,
    name="Disadvantaged Communities",
    style_function=lambda f: {
        'fillColor': '#f28500',
        'color': '#f28500',
        'weight': 0.5,
        'fillOpacity': 0.3,
    }
).add_to(m)

# NYCHA NOT in DAC → blue
for _, row in nycha_gdf[~nycha_gdf['in_dac']].iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='#1a6faf',
        fill=True,
        fill_color='#1a6faf',
        fill_opacity=0.8,
        tooltip=f"{row['development']} ({row['borough']})"
    ).add_to(m)

# NYCHA IN DAC → red
for _, row in nycha_gdf[nycha_gdf['in_dac']].iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='#d62728',
        fill=True,
        fill_color='#d62728',
        fill_opacity=0.9,
        tooltip=f"⚠️ DAC: {row['development']} ({row['borough']})"
    ).add_to(m)

# Legend
legend_html = """
<div style="position:fixed; bottom:40px; left:40px; z-index:1000;
     background:white; padding:12px 16px; border-radius:8px;
     box-shadow:2px 2px 6px rgba(0,0,0,0.3); font-family:Arial; font-size:13px;">
  <b>NYCHA & Disadvantaged Communities</b><br><br>
  <span style="color:#d62728;">●</span> NYCHA in DAC census tract<br>
  <span style="color:#1a6faf;">●</span> NYCHA not in DAC census tract<br>
  <span style="background:#f28500; opacity:0.5; padding:0 8px;">&nbsp;</span> DAC Census Tract
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl().add_to(m)

# ── 7. Save ───────────────────────────────────────────────────────────────────
output_path = base / "outputs/nycha_dac_map.html"
output_path.parent.mkdir(exist_ok=True)
m.save(str(output_path))
print(f"✅ Map saved to {output_path.resolve()}")

ModuleNotFoundError: No module named 'geopandas'

In [8]:
# Check why so few matched
print("Sample NYCHA tract_padded:", sorted(nycha['tract_padded'].unique())[:10])
print("Sample DAC tract_padded:", sorted(list(dac_tracts))[:10])

# Check for non-NYC coordinates
print("\nBorough values:", nycha['borough'].unique())
print("\nLat range:", nycha['latitude'].min(), "to", nycha['latitude'].max())
print("Lon range:", nycha['longitude'].min(), "to", nycha['longitude'].max())

Sample NYCHA tract_padded: ['000006', '000007', '000016', '000018', '000020', '000023', '000024', '000025', '000029', '000035']
Sample DAC tract_padded: ['000100', '000200', '000201', '000202', '000300', '000400', '000600', '000700', '000800', '000900']

Borough values: <StringArray>
['BRONX', 'MANHATTAN', 'BROOKLYN', 'QUEENS', 'STATEN ISLAND']
Length: 5, dtype: str

Lat range: 40.57201 to 40.876983
Lon range: -74.165019 to -73.731469


In [9]:
# Remove commas and convert
nycha['tract_clean'] = nycha['census_tract_(2020)'].astype(str).str.replace(',', '', regex=False)
nycha['tract_census'] = (nycha['tract_clean'].astype(float) * 100).astype(int).astype(str).str.zfill(6)
nycha['geoid_built'] = nycha['county_fips'] + nycha['tract_census']

dac_geoid_set = set(dac_nyc['GEOID'].astype(str))
nycha['in_dac'] = nycha['geoid_built'].isin(dac_geoid_set)

print(f"Total NYCHA buildings:      {len(nycha)}")
print(f"NYCHA buildings in DAC:     {nycha['in_dac'].sum()}")
print(f"NYCHA buildings NOT in DAC: {(~nycha['in_dac']).sum()}")

print("\nSample NYCHA GEOIDs built:", nycha['geoid_built'].head(5).tolist())
print("Sample DAC GEOIDs:        ", list(dac_geoid_set)[:5])

KeyError: 'county_fips'